# DB7-031 — train a hierarchical BRB on all 48 Hudgins features

## Question and agreed model
Can a BRB directly classify the 17 Exercise B gestures from 48 EMG time-domain features?
The user selected a hierarchical design: each of 12 channel modules receives MAV, WL, ZC and SSC.
Every feature has **three reference values**, producing **81 rules per channel and 972 rules total**.
Each rule has 17 consequent beliefs. Channel beliefs are fused with a complete-belief ER calculation.
The model retains all 48 input features. It is not a full 3^48 rule grid and is not an SI-to-I switch.

## What is trained
Adam jointly learns **consequent belief logits, 972 rule weights, 48 attribute weights and 12 channel-fusion weights** through the final gesture cross-entropy loss.
Initial consequent beliefs equal smoothed fitting-class priors; rule and attribute weights start at one; fusion weights start at 1/12.
Rule weights are positive exponentials. Attribute weights are positive and normalized by the maximum within each module.
Fusion weights are positive and sum to one. Consequent beliefs use softmax, so each rule sums to one.
Raw rule/attribute/fusion logits are projected to [-2,2]; belief logits to [-12,12].

## Preserved data protocol
S1–S20; E1 restimulus labels1–17; rest excluded; training repetitions1/3/4/6; test2/5.
200ms windows at2kHz;50ms training stride and10ms test stride, matching DB7-030.
EMG filtering: per-gesture-repetition fourth-order20–450Hz Butterworth followed by50Hz Q30 notch, both zero-phase.
Features are the same 48 MAV/WL/ZC/SSC values. No feature selection or neural retraining.
BRB membership scaling uses training-derived raw feature references, not a new test-fitted scaler.

## Validation and reference values
Four leave-one-training-repetition-out BRB fits per subject/seed select the earliest epoch with highest mean validation accuracy.
Every fold fits feature references from its three fitting repetitions:5th,50th,95th percentiles. Tied references receive a tiny deterministic expansion to remain ordered.
References are fixed during gradient training; rule, attribute and consequent parameters are trained.
For each outer fold, ZC/SSC threshold-ratio screening is nested within those three fitting repetitions using the existing LDA procedure; its amplitude scales also exclude the inner held repetition.
Final BRB training uses all four training repetitions and the verified DB7-030 final threshold choice. Test labels never select epochs, references or thresholds.

## Prespecified optimization and compute
Three genuine BRB seeds42/43/44; up to60 validation epochs; batch512; Adam learning rate0.02; cosine schedule T_max60; regularization0.001; gradient norm limit5.
These are BRB-specific optimization settings, not the earlier CNN's13-epoch schedule.
240 validation BRB fits plus60 final BRB fits. Two processes use cuda:0 and cuda:1 on Kaggle T4x2, dividing subjects between GPUs.
The run aborts if two GPUs are unavailable. A CPU synthetic self-test checks ER equivalence, gradients for every learned parameter group and loss reduction before full fitting.

## Evaluation limits
Primary comparator: matched all48-feature regularized LDA. SI also uses inertial signals, so its accuracy is not a modality-matched comparator.
These recordings were previously inspected; the experiment remains exploratory. Overlapping windows and multiple seeds are not independent trials.
Mean subject accuracy, pooled accuracy, macro-F1, per-gesture errors and paired subject-level uncertainty are reported.
No claim of98% accuracy is made before results.


## 1. Write the unchanged DB7-030 preprocessing implementation

In [ ]:
%%writefile /kaggle/working/db7_030_hudgins.py
"""DB7-030: Hudgins TD4 complementarity against frozen DB7-016 SI/I outputs.

The neural predictions are inputs, never re-fitted. Every TD4 fitting or
selection decision uses repetitions 1/3/4/6; repetitions 2/5 are evaluated
only after those choices are fixed. Overlapping windows stay in their trial.
"""
from __future__ import annotations

import argparse
import hashlib
import itertools
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import butter, filtfilt, iirnotch, sosfiltfilt
from scipy.stats import rankdata

FS = 2000
WINDOW = 400
TRAIN_STEP = 100  # 50 ms: reduce duplicated training evidence
TEST_STEP = 20  # 10 ms: match the frozen SI/I prediction grid
TRAIN_REPS = (1, 3, 4, 6)
TEST_REPS = (2, 5)
FAMILIES = ("MAV", "WL", "ZC", "SSC")
THRESHOLD_RATIOS = (0.005, 0.01, 0.02)
KEY = ["subject", "gesture", "native_repetition", "window_start", "window_end"]


def find_root() -> Path:
    for parent in (Path("/kaggle/input"), Path("D:/NinaProDB7")):
        if parent.exists():
            found = sorted(parent.rglob("S1_E1_A1.mat"))
            if found:
                return found[0].parent.parent
    raise FileNotFoundError("NinaPro DB7 E1 recordings are not attached")


def load_subject(root: Path, subject: int):
    candidates = list(root.rglob(f"S{subject}_E1_A1.mat"))
    if len(candidates) != 1:
        raise ValueError(f"S{subject}: expected one E1 file, found {len(candidates)}")
    data = loadmat(candidates[0], variable_names=["emg", "restimulus", "rerepetition"])
    emg = np.asarray(data["emg"], dtype=np.float32)
    if emg.shape[1] != 12 and emg.shape[0] == 12:
        emg = emg.T
    labels = np.asarray(data["restimulus"]).ravel().astype(int)
    repetitions = np.asarray(data["rerepetition"]).ravel().astype(int)
    if emg.shape[1] != 12 or len(emg) != len(labels) or len(emg) != len(repetitions):
        raise ValueError(f"S{subject}: invalid EMG/annotation alignment")
    if set(np.unique(labels)) != set(range(18)):
        raise ValueError(f"S{subject}: E1 labels 0..17 are not present")
    sos = butter(4, [20, 450], btype="bandpass", fs=FS, output="sos")
    b, a = iirnotch(50, 30, fs=FS)
    edges = np.r_[0, np.flatnonzero(np.diff(labels)) + 1, len(labels)]
    runs = []
    for start, end in zip(edges[:-1], edges[1:]):
        gesture = int(labels[start])
        if gesture == 0:
            continue
        unique_rep = np.unique(repetitions[start:end])
        if len(unique_rep) != 1 or unique_rep[0] not in (*TRAIN_REPS, *TEST_REPS):
            raise ValueError(f"S{subject} gesture {gesture}: invalid repetition")
        filtered = sosfiltfilt(sos, emg[start:end], axis=0)
        filtered = filtfilt(b, a, filtered, axis=0).astype(np.float32)
        runs.append((gesture, int(unique_rep[0]), int(start), filtered))
    counts = pd.Series([(g, r) for g, r, _, _ in runs]).value_counts()
    if len(counts) != 102 or not (counts == 1).all():
        raise ValueError(f"S{subject}: expected 17 gestures x 6 repetitions")
    digest = hashlib.sha256(emg.tobytes()).hexdigest()
    return runs, {"file": str(candidates[0]), "raw_emg_sha256": digest}


def threshold_scale(runs, fitting_repetitions=TRAIN_REPS):
    """Estimate channel amplitudes using only the repetitions used for fitting."""
    train_signal = np.concatenate([np.abs(x) for _, r, _, x in runs
                                   if r in fitting_repetitions])
    return np.maximum(np.percentile(train_signal, 95, axis=0), 1e-12)


def extract(runs, subject, split, threshold):
    features, rows = [], []
    step = TRAIN_STEP if split == "train" else TEST_STEP
    reps = TRAIN_REPS if split == "train" else TEST_REPS
    for gesture, rep, absolute_start, signal in runs:
        if rep not in reps or len(signal) < WINDOW:
            continue
        # Shape: windows x channels x 400 samples. Each run is handled alone.
        windows = np.lib.stride_tricks.sliding_window_view(signal, WINDOW, axis=0)[::step]
        mav = np.abs(windows).mean(axis=-1)
        difference = np.diff(windows, axis=-1)
        wl = np.abs(difference).sum(axis=-1)
        zc = ((windows[..., :-1] * windows[..., 1:] < 0) &
              (np.abs(difference) >= threshold[None, :, None])).sum(axis=-1)
        ssc = ((difference[..., :-1] * difference[..., 1:] < 0) &
               (np.maximum(np.abs(difference[..., :-1]),
                           np.abs(difference[..., 1:])) >= threshold[None, :, None])).sum(axis=-1)
        features.append(np.concatenate([mav, wl, zc, ssc], axis=1).astype(np.float32))
        offsets = np.arange(len(windows)) * step
        rows.extend((subject, gesture, rep, absolute_start + int(k), absolute_start + int(k) + WINDOW)
                    for k in offsets)
    frame = pd.DataFrame(rows, columns=KEY)
    if frame.duplicated(KEY).any():
        raise AssertionError("Duplicate feature windows")
    return np.concatenate(features), frame


def columns(mask):
    return np.concatenate([np.arange(12 * i, 12 * (i + 1)) for i in range(4) if mask & (1 << i)])


def fit_lda(x, y):
    """Train-only robust scaling and regularized pooled-covariance LDA."""
    x = np.asarray(x, dtype=np.float64)
    center = np.median(x, axis=0)
    scale = np.percentile(x, 75, axis=0) - np.percentile(x, 25, axis=0)
    scale = np.maximum(scale, np.maximum(np.max(np.abs(x), axis=0) * 1e-6, 1e-9))
    z = (x - center) / scale
    classes = np.arange(1, 18)
    counts = np.bincount(y, minlength=18)[1:]
    if np.any(counts == 0):
        raise ValueError("LDA training is missing a gesture")
    means = np.stack([z[y == g].mean(axis=0) for g in classes])
    residual = z - means[y - 1]
    covariance = residual.T @ residual / max(len(z) - len(classes), 1)
    diagonal = np.diag(np.diag(covariance))
    covariance = 0.9 * covariance + 0.1 * diagonal
    covariance.flat[::len(covariance) + 1] += 1e-5
    weights = np.linalg.solve(covariance, means.T)
    bias = -0.5 * np.sum(means * weights.T, axis=1) + np.log(counts / counts.sum())
    return center, scale, weights, bias


def predict(model, x):
    center, scale, weights, bias = model
    score = ((np.asarray(x, dtype=np.float64) - center) / scale) @ weights + bias
    score -= score.max(axis=1, keepdims=True)
    prob = np.exp(score)
    prob /= prob.sum(axis=1, keepdims=True)
    return prob.argmax(axis=1).astype(np.int16) + 1, prob


def cv_score(fold_features, frame, mask):
    """Each fold has features extracted with its own fitting-only threshold."""
    y = frame.gesture.to_numpy(int)
    rep = frame.native_repetition.to_numpy(int)
    ix = columns(mask)
    scores = []
    for held in TRAIN_REPS:
        train, val = rep != held, rep == held
        x = fold_features[held]
        pred, _ = predict(fit_lda(x[train][:, ix], y[train]), x[val][:, ix])
        scores.append(float(np.mean(pred == y[val])))
    return float(np.mean(scores))


def check_trace_coverage(subject_trace, test_meta, subject):
    """Require exactly one prediction per raw test window for each saved seed."""
    if set(subject_trace.seed.unique()) != {42, 43, 44}:
        raise AssertionError(f"S{subject}: expected exactly seeds 42, 43 and 44")
    expected = pd.MultiIndex.from_frame(test_meta[KEY])
    for seed, group in subject_trace.groupby("seed"):
        actual = pd.MultiIndex.from_frame(group[KEY])
        if actual.has_duplicates or len(actual) != len(expected):
            raise AssertionError(f"S{subject} seed {seed}: duplicate/missing test windows")
        if len(expected.difference(actual)) or len(actual.difference(expected)):
            raise AssertionError(f"S{subject} seed {seed}: raw and saved test keys differ")


def subset_complementarity(joined, prediction, margin):
    """Count useful TD4 evidence and harmful switches for one subject and seed."""
    truth = joined.gesture.to_numpy(int)
    si = joined.si_pred.to_numpy(int)
    inertial = joined.i_pred.to_numpy(int)
    si_correct, i_correct = si == truth, inertial == truth
    good = prediction == truth
    shared = ~si_correct & ~i_correct
    recovery, harm = ~si_correct & i_correct, si_correct & ~i_correct
    switch = (si != inertial) & (margin > 0)
    recovered, harmed = int((switch & recovery).sum()), int((switch & harm).sum())
    return dict(windows=len(truth), td_correct=int(good.sum()),
        td_accuracy=float(good.mean()), si_correct=int(si_correct.sum()),
        si_accuracy=float(si_correct.mean()), shared_wrong=int(shared.sum()),
        shared_recovered=int((shared & good).sum()),
        si_wrong_td_correct=int((~si_correct & good).sum()),
        si_correct_td_wrong=int((si_correct & ~good).sum()),
        recovery_opportunities=int(recovery.sum()), harm_opportunities=int(harm.sum()),
        td_correct_on_recovery=int((recovery & good).sum()),
        td_correct_on_harm=int((harm & good).sum()),
        switches=int(switch.sum()), switch_recovered=recovered, switch_harmed=harmed,
        switch_net_corrected=recovered - harmed,
        switch_accuracy=float((si_correct.sum() + recovered - harmed) / len(truth)),
        si_i_td_oracle_correct=int((si_correct | i_correct | good).sum()))


def subject_study(root, subject, trace, out):
    runs, source = load_subject(root, subject)
    scale = threshold_scale(runs)
    # A validation repetition must not affect the ZC/SSC amplitude thresholds.
    fold_scales = {held: threshold_scale(runs, tuple(r for r in TRAIN_REPS if r != held))
                   for held in TRAIN_REPS}
    threshold_cv = []
    training = {}
    cv_features = {}
    for ratio in THRESHOLD_RATIOS:
        x, meta = extract(runs, subject, "train", scale * ratio)
        training[ratio] = x
        cv_features[ratio] = {}
        for held in TRAIN_REPS:
            fold_x, fold_meta = extract(runs, subject, "train", fold_scales[held] * ratio)
            if not fold_meta.equals(meta):
                raise AssertionError("Fold feature windows changed with the noise threshold")
            cv_features[ratio][held] = fold_x
        score = cv_score(cv_features[ratio], meta, 15)
        threshold_cv.append(dict(subject=subject, ratio=ratio, td4_cv_accuracy=score))
    selected_ratio = sorted(THRESHOLD_RATIOS,
                            key=lambda v: (-next(r["td4_cv_accuracy"] for r in threshold_cv if r["ratio"] == v),
                                           abs(v - 0.01)))[0]
    x_train = training[selected_ratio]
    selected_cv_features = cv_features[selected_ratio]
    del training
    del cv_features
    x_test, test_meta = extract(runs, subject, "test", scale * selected_ratio)
    y_train = meta.gesture.to_numpy(int)
    y_test = test_meta.gesture.to_numpy(int)
    subset_cv = []
    for mask in range(1, 16):
        subset_cv.append(dict(subject=subject, mask=mask,
                              families="+".join(FAMILIES[i] for i in range(4) if mask & (1 << i)),
                              cv_accuracy=cv_score(selected_cv_features, meta, mask)))
    del selected_cv_features
    chosen = sorted(subset_cv, key=lambda r: (-r["cv_accuracy"], int(r["mask"]).bit_count(), r["mask"]))[0]["mask"]
    test_rows, predictions = [], {}
    for row in subset_cv:
        mask = row["mask"]
        ix = columns(mask)
        pred, prob = predict(fit_lda(x_train[:, ix], y_train), x_test[:, ix])
        predictions[mask] = (pred, prob)
        test_rows.append({**row, "test_accuracy": float(np.mean(pred == y_test)),
                          "correct": int(np.sum(pred == y_test)), "windows": len(y_test),
                          "cv_selected": mask == chosen, "td4_full": mask == 15,
                          "selected_threshold_ratio": selected_ratio})
    subject_trace = trace[trace.subject == subject].copy()
    check_trace_coverage(subject_trace, test_meta, subject)
    joined = subject_trace.merge(test_meta.assign(feature_row=np.arange(len(test_meta))),
                                 on=KEY, how="left", validate="many_to_one")
    if joined.feature_row.isna().any() or len(joined) != len(subject_trace):
        raise AssertionError(f"S{subject}: saved neural/test feature window mismatch")
    if not np.array_equal(joined.gesture.to_numpy(), y_test[joined.feature_row.to_numpy(int)]):
        raise AssertionError("Label mismatch after joining predictions")
    combination_rows = []
    indices = joined.feature_row.to_numpy(int)
    si, inertial = joined.si_pred.to_numpy(int), joined.i_pred.to_numpy(int)
    for row in subset_cv:
        mask = row["mask"]
        pred, prob = predictions[mask]
        aligned_pred = pred[indices]
        margin = prob[indices, inertial - 1] - prob[indices, si - 1]
        for seed, positions in joined.groupby("seed").indices.items():
            positions = np.asarray(positions)
            combination_rows.append(dict(subject=subject, seed=int(seed), mask=mask,
                families=row["families"], cv_selected=mask == chosen,
                **subset_complementarity(joined.iloc[positions], aligned_pred[positions],
                                         margin[positions])))
    for name, mask in (("selected", chosen), ("TD4", 15)):
        pred, prob = predictions[mask]
        indices = joined.feature_row.to_numpy(int)
        si = joined.si_pred.to_numpy(int)
        inertial = joined.i_pred.to_numpy(int)
        joined[f"{name}_pred"] = pred[indices]
        joined[f"{name}_correct"] = pred[indices] == joined.gesture.to_numpy(int)
        # Candidate support is diagnostic; do not treat raw LDA probabilities as calibrated.
        joined[f"{name}_candidate_margin"] = (prob[indices, inertial - 1] -
                                               prob[indices, si - 1]).astype(np.float32)
    joined.drop(columns="feature_row").to_csv(out / f"S{subject:02}_window_predictions.csv.gz", index=False)
    (out / f"S{subject:02}_source.json").write_text(json.dumps({**source, "subject": subject,
        "threshold_ratio": selected_ratio, "selected_subset": chosen,
        "threshold_scale_final": scale.tolist(),
        "threshold_scale_by_cv_held_repetition": {str(r): v.tolist() for r, v in fold_scales.items()},
        "train_windows_50ms": len(x_train), "test_windows_10ms": len(x_test)}, indent=2))
    print(f"S{subject:02}: train {len(x_train)}, test {len(x_test)}, "
          f"TD4 {test_rows[14]['test_accuracy']:.4f}, chosen {chosen:04b}", flush=True)
    return threshold_cv, subset_cv, test_rows, combination_rows


def summary(out, subjects):
    parts = [pd.read_csv(out / f"S{s:02}_window_predictions.csv.gz") for s in subjects]
    data = pd.concat(parts, ignore_index=True)
    if data.duplicated(KEY + ["seed"]).any():
        raise AssertionError("Duplicate test predictions")
    expected = len(data)
    target = data.gesture.to_numpy()
    correct = {"SI": data.si_pred.to_numpy() == target, "I": data.i_pred.to_numpy() == target,
               "TD4": data.TD4_pred.to_numpy() == target,
               "selected": data.selected_pred.to_numpy() == target}
    si_wrong = ~correct["SI"]
    shared = si_wrong & ~correct["I"]
    recovery = si_wrong & correct["I"]
    prevent_harm = correct["SI"] & ~correct["I"]
    records = []
    gate_rows = []
    for method in ("TD4", "selected"):
        good = correct[method]
        records.append(dict(method=method, windows=expected, correct=int(good.sum()),
            accuracy=float(good.mean()), si_errors=int(si_wrong.sum()),
            si_i_both_wrong=int(shared.sum()), shared_errors_recovered=int((shared & good).sum()),
            shared_errors_recovery_pct=float(100 * (shared & good).sum() / shared.sum()),
            si_wrong_i_correct_support=int((recovery & good).sum()),
            si_correct_i_wrong_td_correct=int((prevent_harm & good).sum()),
            si_correct_td_wrong=int((correct["SI"] & ~good).sum()),
            si_i_td_oracle_accuracy=float((correct["SI"] | correct["I"] | good).mean()),
            si_td_oracle_accuracy=float((correct["SI"] | good).mean())))
        margin = data[f"{method}_candidate_margin"].to_numpy(float)
        switch = (data.si_pred.to_numpy() != data.i_pred.to_numpy()) & (margin > 0)
        recovered = int((switch & recovery).sum())
        harmed = int((switch & prevent_harm).sum())
        directional = recovery | prevent_harm
        ranks = rankdata(margin[directional], method="average")
        positive = recovery[directional]
        negative_count = int((~positive).sum())
        auc = ((ranks[positive].sum() - positive.sum() * (positive.sum() + 1) / 2)
               / (positive.sum() * negative_count))
        gate_rows.append(dict(method=method, disagreement_windows=int((data.si_pred != data.i_pred).sum()),
            recovery_opportunities=int(recovery.sum()), harm_opportunities=int(prevent_harm.sum()),
            pair_margin_auc_recovery_vs_harm=float(auc),
            fixed_zero_margin_switches=int(switch.sum()), recovered=recovered, harmed=harmed,
            net_corrected=recovered - harmed,
            fixed_zero_margin_gate_accuracy=float((correct["SI"].sum() + recovered - harmed) / expected),
            note="Fixed threshold zero; diagnostic only, no BRB fitting or threshold tuning"))
    pd.DataFrame(records).to_csv(out / "complementarity_summary.csv", index=False)
    pd.DataFrame(gate_rows).to_csv(out / "candidate_preference_diagnostic.csv", index=False)
    for grouping, filename in [(["subject", "seed"], "subject_seed_complementarity.csv"),
                               (["subject", "gesture"], "subject_gesture_complementarity.csv"),
                               (["gesture"], "gesture_complementarity.csv"),
                               (["phase"], "phase_complementarity.csv")]:
        rows = []
        for key, indices in data.groupby(grouping, sort=True).indices.items():
            ix = np.asarray(indices)
            values = key if isinstance(key, tuple) else (key,)
            row = dict(zip(grouping, values))
            row.update(windows=len(ix), SI_accuracy=float(correct["SI"][ix].mean()),
                       I_accuracy=float(correct["I"][ix].mean()))
            for method in ("TD4", "selected"):
                row[f"{method}_accuracy"] = float(correct[method][ix].mean())
                row[f"{method}_shared_recovered"] = int((shared & correct[method])[ix].sum())
                row[f"{method}_si_error_recovered"] = int((si_wrong & correct[method])[ix].sum())
                row[f"{method}_si_correct_harmed"] = int((correct["SI"] & ~correct[method])[ix].sum())
            row["SI_wrong"] = int(si_wrong[ix].sum())
            row["shared_wrong"] = int(shared[ix].sum())
            rows.append(row)
        pd.DataFrame(rows).to_csv(out / filename, index=False)
    selected = pd.read_csv(out / "subset_test_metrics.csv")
    aggregate = selected.groupby(["mask", "families"]).agg(mean_subject_test_accuracy=("test_accuracy", "mean"),
        mean_subject_cv_accuracy=("cv_accuracy", "mean"), subjects_selected=("cv_selected", "sum")).reset_index()
    aggregate.sort_values("mean_subject_cv_accuracy", ascending=False).to_csv(out / "feature_combination_summary.csv", index=False)
    combinations = pd.read_csv(out / "subset_subject_seed_complementarity.csv")
    count_fields = ["windows", "td_correct", "si_correct", "shared_wrong", "shared_recovered",
        "si_wrong_td_correct", "si_correct_td_wrong", "recovery_opportunities", "harm_opportunities",
        "td_correct_on_recovery", "td_correct_on_harm", "switches", "switch_recovered", "switch_harmed",
        "switch_net_corrected", "si_i_td_oracle_correct"]
    rows = []
    for (mask, families), group in combinations.groupby(["mask", "families"]):
        row = dict(mask=int(mask), families=families,
                   **{field: int(group[field].sum()) for field in count_fields})
        row.update(mean_subject_td_accuracy=float(group.td_accuracy.mean()),
            mean_subject_si_accuracy=float(group.si_accuracy.mean()),
            mean_subject_switch_accuracy=float(group.switch_accuracy.mean()),
            pooled_td_accuracy=row["td_correct"] / row["windows"],
            pooled_switch_accuracy=(row["si_correct"] + row["switch_net_corrected"]) / row["windows"],
            shared_recovery_pct=100 * row["shared_recovered"] / max(row["shared_wrong"], 1),
            pooled_si_i_td_oracle_accuracy=row["si_i_td_oracle_correct"] / row["windows"])
        rows.append(row)
    pd.DataFrame(rows).to_csv(out / "subset_complementarity_summary.csv", index=False)
    completion = dict(success=True, subjects=subjects, seeds=[42, 43, 44],
        test_window_seed_evaluations=expected, neural_fits=0,
        lda_final_fits=15 * len(subjects), lda_cv_fits=(3 + 15) * 4 * len(subjects),
        exact_seed_window_coverage_verified=True, cv_threshold_scale_fitting_repetitions_only=True,
        protocol="DB7-016 200ms/10ms test; 50ms TD4 training grid; repetition-grouped train-only fourfold selection")
    (out / "completion.json").write_text(json.dumps(completion, indent=2))
    return completion


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", type=Path, default=None)
    parser.add_argument("--trace", type=Path, required=True)
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--subjects", type=int, nargs="+", default=list(range(1, 21)))
    args = parser.parse_args()
    root = args.input or find_root()
    args.out.mkdir(parents=True, exist_ok=True)
    trace = pd.read_csv(args.trace)
    needed = KEY + ["seed", "si_pred", "i_pred", "phase"]
    if any(v not in trace for v in needed):
        raise ValueError("Saved SI/I trace is missing required columns")
    if trace.duplicated(KEY + ["seed"]).any():
        raise ValueError("Saved SI/I trace has duplicate windows")
    source = dict(raw_root=str(root), trace_file=str(args.trace),
                  trace_sha256=hashlib.sha256(args.trace.read_bytes()).hexdigest(),
                  subjects=args.subjects, train_repetitions=TRAIN_REPS,
                  test_repetitions=TEST_REPS, window_samples=WINDOW,
                  train_step_samples=TRAIN_STEP, test_step_samples=TEST_STEP,
                  feature_families=FAMILIES, threshold_ratios=THRESHOLD_RATIOS,
                  lda_covariance_shrinkage=0.1,
                  cv_threshold_scale="95th percentile fitted separately on three fitting repetitions",
                  final_threshold_scale="95th percentile fitted on repetitions 1/3/4/6",
                  testing_note="Existing inspected DB7-016 test split; exploratory, no independent confirmation")
    (args.out / "PROTOCOL.json").write_text(json.dumps(source, indent=2))
    threshold_rows, cv_rows, test_rows, combination_rows = [], [], [], []
    for s in args.subjects:
        a, b, c, d = subject_study(root, s, trace, args.out)
        threshold_rows.extend(a)
        cv_rows.extend(b)
        test_rows.extend(c)
        combination_rows.extend(d)
        pd.DataFrame(threshold_rows).to_csv(args.out / "threshold_cv.csv", index=False)
        pd.DataFrame(cv_rows).to_csv(args.out / "subset_cv.csv", index=False)
        pd.DataFrame(test_rows).to_csv(args.out / "subset_test_metrics.csv", index=False)
        pd.DataFrame(combination_rows).to_csv(args.out / "subset_subject_seed_complementarity.csv", index=False)
    print(json.dumps(summary(args.out, args.subjects), indent=2), flush=True)


if __name__ == "__main__":
    main()


## 2. Write the trainable hierarchical BRB implementation
The implementation below exposes membership matching, ER aggregation, training, rule export, validation and evaluation as separate functions.

In [ ]:
%%writefile /kaggle/working/db7_031_brb.py
"""DB7-031: train a hierarchical 48-input, three-reference BRB classifier.

Twelve channel BRBs each contain 3^4=81 rules. Their 17-class beliefs are
combined using ER. Consequents, rule/attribute weights and fusion weights
are jointly optimized. Test repetitions never select parameters or epochs.
"""
from pathlib import Path
import os
os.environ.setdefault('OPENBLAS_NUM_THREADS','1')
os.environ.setdefault('OMP_NUM_THREADS','1')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import argparse, itertools, json, time, subprocess, sys, zipfile, hashlib
import numpy as np
import pandas as pd
import db7_030_hudgins as features

SEEDS=(42,43,44)
MAX_EPOCHS=60
BATCH=512
LR=.02
PENALTY=.001
NAMES=[f'{f}_CH{c:02}' for f in features.FAMILIES for c in range(1,13)]
BITS=np.array(list(itertools.product(range(3),repeat=4)),dtype=np.int64)
ACTIVE_BITS=np.array(list(itertools.product(range(2),repeat=4)),dtype=np.int64)

def write(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,indent=2,allow_nan=False),encoding='utf-8')

def reference_values(x):
    """Three raw-feature references, fitted using fitting repetitions only."""
    r=np.percentile(x,[5,50,95],axis=0).T
    span=np.maximum(r[:,2]-r[:,0],np.maximum(np.max(np.abs(x),axis=0)*1e-6,1e-12))
    tiny=span*1e-6
    r[:,0]=np.minimum(r[:,0],r[:,1]-tiny)
    r[:,2]=np.maximum(r[:,2],r[:,1]+tiny)
    assert r.shape==(48,3) and np.all(np.diff(r,axis=1)>0)
    return r

def encode(x,refs):
    """Only 16 of 81 rules can activate per channel; retain matching degrees."""
    x=np.asarray(x,float).reshape(-1,4,12).transpose(0,2,1)
    r=refs.reshape(4,12,3).transpose(1,0,2)
    low=(x>=r[None,:,:,1]).astype(np.int64)
    a=np.where(low==0,r[None,:,:,0],r[None,:,:,1])
    b=np.where(low==0,r[None,:,:,1],r[None,:,:,2])
    frac=np.clip((x-a)/(b-a),0,1)
    matches=np.where(ACTIVE_BITS[None,None,:,:]==0,1-frac[:,:,None,:],frac[:,:,None,:])
    index=((low[:,:,None,:]+ACTIVE_BITS[None,None,:,:])*np.array([27,9,3,1])).sum(-1)
    valid=np.all(matches>0,axis=-1)
    assert valid.any(axis=-1).all()
    return index,np.log(np.maximum(matches,1e-30)).astype(np.float32),valid

def er_numpy(a,beta,axis):
    product=np.prod(1+a[...,None]*(beta-1),axis=axis)
    ignorance=np.prod(1-a,axis=axis)
    v=product-ignorance[...,None]
    return v/v.sum(-1,keepdims=True)

def make_model(prior):
    import torch
    from torch import nn
    class HierarchicalBRB(nn.Module):
        def __init__(self):
            super().__init__()
            initial=torch.tensor(np.log(prior),dtype=torch.float32).repeat(12,81,1)
            self.belief_logits=nn.Parameter(initial.clone())
            self.register_buffer('initial_logits',initial)
            self.rule_logits=nn.Parameter(torch.zeros(12,81))
            self.attribute_logits=nn.Parameter(torch.zeros(12,4))
            self.fusion_logits=nn.Parameter(torch.zeros(12))
        def weights(self):
            aw=self.attribute_logits.exp()
            aw=aw/aw.amax(dim=-1,keepdim=True)
            return self.belief_logits.softmax(-1),self.rule_logits.exp(),aw,self.fusion_logits.softmax(-1)
        @staticmethod
        def er(a,beta,dim):
            v=torch.prod(1+a.unsqueeze(-1)*(beta-1),dim=dim)-torch.prod(1-a,dim=dim).unsqueeze(-1)
            v=v.clamp_min(1e-12)
            return v/v.sum(-1,keepdim=True)
        def forward(self,index,logmatches,valid):
            beta,rw,aw,fw=self.weights()
            channel=torch.arange(12,device=index.device)[None,:,None]
            score=(logmatches*aw[None,:,None,:]).sum(-1)+self.rule_logits[channel,index]
            activation=score.masked_fill(~valid,-torch.inf).softmax(-1)
            channel_beliefs=self.er(activation,beta[channel,index],dim=2)
            fusion=fw[None,:].expand(len(index),-1)
            return self.er(fusion,channel_beliefs,dim=1)
        def regularizer(self):
            return PENALTY*((self.belief_logits-self.initial_logits).square().mean()+
                self.rule_logits.square().mean()+self.attribute_logits.square().mean()+self.fusion_logits.square().mean())
    return HierarchicalBRB()

def tensors(x,refs,device):
    import torch
    idx,logm,valid=encode(x,refs)
    return (torch.as_tensor(idx,device=device,dtype=torch.long),
            torch.as_tensor(logm,device=device),torch.as_tensor(valid,device=device))

def probabilities(model,data):
    import torch
    model.eval();result=[]
    with torch.no_grad():
        for start in range(0,len(data[0]),BATCH):
            result.append(model(*(v[start:start+BATCH] for v in data)).cpu().numpy())
    p=np.concatenate(result)
    assert np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=2e-6)
    return p

def fit(x,y,refs,seed,epochs,device,val=None):
    import torch
    torch.manual_seed(seed)
    if str(device).startswith('cuda'):torch.cuda.manual_seed_all(seed)
    prior=np.bincount(y,minlength=17).astype(float)+.5
    prior/=prior.sum()
    model=make_model(prior).to(device)
    initial={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    train=tensors(x,refs,device);target=torch.as_tensor(y,device=device,dtype=torch.long)
    validation=tensors(val[0],refs,device) if val is not None else None
    optimizer=torch.optim.Adam(model.parameters(),lr=LR)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=MAX_EPOCHS)
    history=[]
    for epoch in range(1,epochs+1):
        model.train();order=torch.randperm(len(y),device=device);total=0.
        for begin in range(0,len(y),BATCH):
            ix=order[begin:begin+BATCH]
            optimizer.zero_grad(set_to_none=True)
            p=model(*(v[ix] for v in train))
            loss=-p[torch.arange(len(ix),device=device),target[ix]].clamp_min(1e-12).log().mean()+model.regularizer()
            if not torch.isfinite(loss):raise RuntimeError('Nonfinite training loss')
            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),5.)
            optimizer.step()
            with torch.no_grad():
                model.belief_logits.clamp_(-12,12)
                model.rule_logits.clamp_(-2,2)
                model.attribute_logits.clamp_(-2,2)
                model.fusion_logits.clamp_(-2,2)
            total+=float(loss.detach())*len(ix)
        row={'epoch':epoch,'training_objective':total/len(y),'learning_rate':optimizer.param_groups[0]['lr']}
        if validation is not None:
            pred=probabilities(model,validation).argmax(1)
            row['validation_accuracy']=float(np.mean(pred==val[1]))
        history.append(row);scheduler.step()
    return model,initial,history

def select_ratio(runs,allowed):
    """Nested LDA threshold screening: outer validation rep never participates."""
    scores=[]
    for ratio in features.THRESHOLD_RATIOS:
        fold_scores=[]
        for held in allowed:
            fitting=tuple(r for r in allowed if r!=held)
            scale=features.threshold_scale(runs,fitting)
            x,meta=features.extract(runs,0,'train',scale*ratio)
            rep=meta.native_repetition.to_numpy();y=meta.gesture.to_numpy(int)
            train=np.isin(rep,fitting);validation=rep==held
            pred,_=features.predict(features.fit_lda(x[train],y[train]),x[validation])
            fold_scores.append(float(np.mean(pred==y[validation])))
        scores.append({'ratio':ratio,'mean_accuracy':float(np.mean(fold_scores)),'fold_scores':fold_scores})
    chosen=sorted(scores,key=lambda r:(-r['mean_accuracy'],abs(r['ratio']-.01)))[0]['ratio']
    return chosen,scores

def metrics(y,p):
    pred=p.argmax(1);cm=np.bincount(y*17+pred,minlength=289).reshape(17,17)
    diag=np.diag(cm);rows=cm.sum(1);cols=cm.sum(0)
    recall=np.divide(diag,rows,out=np.zeros(17,float),where=rows>0)
    f1=np.divide(2*diag,rows+cols,out=np.zeros(17,float),where=(rows+cols)>0)
    return {'accuracy':float(np.mean(pred==y)),'macro_f1':float(f1.mean()),
            'balanced_accuracy':float(recall.mean()),'windows':len(y)},cm

def export_rules(folder,model,initial,refs):
    import torch
    final={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    rows=[];fusion=[]
    for state,weights in [('initial',initial),('trained',final)]:
        model.load_state_dict(weights)
        beta,rw,aw,fw=[v.detach().cpu().numpy() for v in model.weights()]
        for c in range(12):
            fusion.append({'state':state,'channel':c+1,'fusion_weight':float(fw[c])})
            for r,bits in enumerate(BITS):
                row={'state':state,'channel':c+1,'rule':r+1,'rule_weight':float(rw[c,r])}
                for f,name in enumerate(features.FAMILIES):
                    row[f'{name}_level']=('low','medium','high')[bits[f]]
                    row[f'{name}_reference']=float(refs[f*12+c,bits[f]])
                    row[f'{name}_attribute_weight']=float(aw[c,f])
                row.update({f'belief_G{k+1}':float(beta[c,r,k]) for k in range(17)})
                rows.append(row)
    pd.DataFrame(rows).to_csv(folder/'initial_trained_rules.csv.gz',index=False)
    pd.DataFrame(fusion).to_csv(folder/'initial_trained_fusion_weights.csv',index=False)
    pd.DataFrame(refs,columns=['low','medium','high']).assign(feature=NAMES).to_csv(folder/'reference_values.csv',index=False)
    model.load_state_dict(final)
    torch.save({'state_dict':final,'initial_state_dict':initial,'references':refs.tolist()},folder/'model.pt')
    return {k:float((final[k]-initial[k]).abs().max()) for k in ['belief_logits','rule_logits','attribute_logits','fusion_logits']}

def run_subject(raw,subject,out,parent,device,max_epochs=MAX_EPOCHS,seeds=SEEDS):
    import torch
    folder=out/f'S{subject:02}';folder.mkdir(parents=True,exist_ok=True)
    runs,raw_source=features.load_subject(raw,subject)
    old=parent[str(subject)]
    assert raw_source['raw_emg_sha256']==old['raw_emg_sha256'],'Raw input differs from DB7-030'
    prepared=[];audit=[]
    for held in features.TRAIN_REPS:
        allowed=tuple(r for r in features.TRAIN_REPS if r!=held)
        ratio,screen=select_ratio(runs,allowed)
        scale=features.threshold_scale(runs,allowed)
        x,meta=features.extract(runs,subject,'train',scale*ratio)
        rep=meta.native_repetition.to_numpy();y=meta.gesture.to_numpy(int)-1
        ti=np.isin(rep,allowed);vi=rep==held
        refs=reference_values(x[ti])
        prepared.append((x[ti],y[ti],x[vi],y[vi],refs))
        audit.append({'held_repetition':held,'fitting_repetitions':allowed,'threshold_ratio':ratio,
                      'nested_threshold_scores':screen,'threshold_scale':scale.tolist(),
                      'reference_values':refs.tolist(),'train_windows':int(ti.sum()),'validation_windows':int(vi.sum())})
        print(f'S{subject:02} {device}: prepared held repetition {held}',flush=True)
    scale=features.threshold_scale(runs)
    assert np.allclose(scale,old['threshold_scale_final'],rtol=1e-10,atol=1e-15)
    xtrain,train_meta=features.extract(runs,subject,'train',scale*old['threshold_ratio'])
    ytrain=train_meta.gesture.to_numpy(int)-1
    xtest,test_meta=features.extract(runs,subject,'test',scale*old['threshold_ratio'])
    ytest=test_meta.gesture.to_numpy(int)-1
    assert len(xtest)==old['test_windows_10ms']
    refs=reference_values(xtrain)
    write(folder/'fold_audit.json',{'raw_source':raw_source,'folds':audit,
          'final_threshold_ratio':old['threshold_ratio'],'final_fitting_repetitions':features.TRAIN_REPS,
          'test_repetitions':features.TEST_REPS,'test_used_for_selection':False})
    baseline_model=features.fit_lda(xtrain,ytrain+1)
    _,lda_p=features.predict(baseline_model,xtest)
    lda_metric,lda_cm=metrics(ytest,lda_p)
    assert abs(lda_metric['accuracy']-old['lda_test_accuracy'])<1e-12,'LDA input parity failed'
    rows=[]
    for seed in seeds:
        seed_dir=folder/f'seed{seed}';seed_dir.mkdir(exist_ok=True)
        cv=[];curves=[]
        for held,(x,y,v,vy,r) in zip(features.TRAIN_REPS,prepared):
            model,_,history=fit(x,y,r,seed,max_epochs,device,val=(v,vy))
            curves.append([h['validation_accuracy'] for h in history])
            cv.extend({'held_repetition':held,**h} for h in history)
            del model
        mean_curve=np.mean(curves,axis=0)
        selected=int(np.argmax(mean_curve))+1  # tie: earliest epoch
        pd.DataFrame(cv).to_csv(seed_dir/'validation_history.csv',index=False)
        model,initial,history=fit(xtrain,ytrain,refs,seed,selected,device)
        pd.DataFrame(history).to_csv(seed_dir/'training_history.csv',index=False)
        test_data=tensors(xtest,refs,device)
        p=probabilities(model,test_data)
        result,cm=metrics(ytest,p)
        train_accuracy=float(np.mean(probabilities(model,tensors(xtrain,refs,device)).argmax(1)==ytrain))
        changes=export_rules(seed_dir,model,initial,refs)
        assert all(v>0 for v in changes.values()),'A requested parameter group did not update'
        predictions=test_meta.copy();predictions['seed']=seed
        predictions['BRB_pred']=p.argmax(1)+1;predictions['LDA_pred']=lda_p.argmax(1)+1
        predictions.to_csv(seed_dir/'predictions.csv.gz',index=False)
        np.savez_compressed(seed_dir/'probabilities.npz',brb=p,lda=lda_p.astype(np.float32))
        pd.DataFrame(cm,index=np.arange(1,18),columns=np.arange(1,18)).to_csv(seed_dir/'confusion.csv',index_label='true_gesture')
        pd.DataFrame({'gesture':np.arange(1,18),'windows':cm.sum(1),'correct':np.diag(cm),
                      'wrong':cm.sum(1)-np.diag(cm),'recall':np.diag(cm)/cm.sum(1)}).to_csv(seed_dir/'gesture_metrics.csv',index=False)
        result.update(subject=subject,seed=seed,selected_epoch=selected,validation_accuracy=float(mean_curve[selected-1]),
                      train_accuracy=train_accuracy,lda_accuracy=lda_metric['accuracy'],device=device,
                      gpu_name=torch.cuda.get_device_name(device) if str(device).startswith('cuda') else 'CPU',
                      parameter_changes=changes,success=True)
        write(seed_dir/'completion.json',result)
        rows.append({k:v for k,v in result.items() if k!='parameter_changes'})
        print(f'S{subject:02} seed{seed} {device}: epoch={selected}; BRB={result["accuracy"]:.5f}; LDA={lda_metric["accuracy"]:.5f}',flush=True)
        del model,test_data
    return rows

def self_test():
    import torch
    torch.set_num_threads(1)
    rng=np.random.default_rng(42)
    x=rng.normal(size=(85,48)).astype(np.float32);y=np.arange(85)%17
    x+=y[:,None]*.3
    refs=reference_values(x);enc=encode(x,refs)
    assert enc[0].shape==(85,12,16)
    prior=np.ones(17)/17;model=make_model(prior).double()
    with torch.no_grad():
        model.belief_logits.add_(torch.tensor(rng.normal(0,.3,(12,81,17))))
        model.rule_logits.add_(torch.tensor(rng.normal(0,.1,(12,81))))
        model.attribute_logits.add_(torch.tensor(rng.normal(0,.1,(12,4))))
    td=(torch.tensor(enc[0][:3]),torch.tensor(enc[1][:3],dtype=torch.float64),torch.tensor(enc[2][:3]))
    p=model(*td);beta,rw,aw,fw=[v.detach().numpy() for v in model.weights()]
    score=(enc[1][:3]*aw[None,:,None,:]).sum(-1)+np.log(rw[np.arange(12)[None,:,None],enc[0][:3]])
    score=np.where(enc[2][:3],score,-np.inf);a=np.exp(score-score.max(-1,keepdims=True));a/=a.sum(-1,keepdims=True)
    channels=er_numpy(a,beta[np.arange(12)[None,:,None],enc[0][:3]],2)
    expected=er_numpy(np.broadcast_to(fw,(3,12)),channels,1)
    assert np.allclose(p.detach().numpy(),expected,atol=1e-10)
    loss=-p[:,0].log().mean();loss.backward()
    for name in ['belief_logits','rule_logits','attribute_logits','fusion_logits']:
        parameter=getattr(model,name);grad=parameter.grad.flatten()
        ix=int(grad.abs().argmax());analytical=float(grad[ix]);eps=1e-5
        with torch.no_grad():parameter.view(-1)[ix]+=eps
        plus=float(-model(*td)[:,0].log().mean().detach())
        with torch.no_grad():parameter.view(-1)[ix]-=2*eps
        minus=float(-model(*td)[:,0].log().mean().detach())
        with torch.no_grad():parameter.view(-1)[ix]+=eps
        assert np.isclose(analytical,(plus-minus)/(2*eps),rtol=2e-3,atol=1e-7),name
    learned,initial,history=fit(x,y,refs,42,8,'cpu',val=(x,y))
    assert history[-1]['training_objective']<history[0]['training_objective']
    for name in ['belief_logits','rule_logits','attribute_logits','fusion_logits']:
        assert float((learned.state_dict()[name]-initial[name]).abs().max())>0,name
    print('SELF TEST PASSED: ER equivalence, all four gradient groups, learning loss, weights updated',flush=True)

def summarize(out):
    rows=[];gestures=[];devices=set()
    for subject in range(1,21):
        for seed in SEEDS:
            folder=out/f'S{subject:02}'/f'seed{seed}'
            r=json.loads((folder/'completion.json').read_text())
            assert r['success'] and all(v>0 for v in r['parameter_changes'].values())
            devices.add(r['device']);rows.append({k:v for k,v in r.items() if k!='parameter_changes'})
            gestures.append(pd.read_csv(folder/'gesture_metrics.csv').assign(subject=subject,seed=seed))
    df=pd.DataFrame(rows);df.to_csv(out/'subject_seed_metrics.csv',index=False)
    pd.concat(gestures).to_csv(out/'subject_gesture_metrics.csv',index=False)
    means=df.groupby('subject')[['accuracy','lda_accuracy']].mean()
    delta=(means.accuracy-means.lda_accuracy).to_numpy()*100
    rng=np.random.default_rng(42);boot=delta[rng.integers(0,20,(20000,20))].mean(1)
    signs=rng.choice([-1,1],(100000,20));null=np.abs((signs*delta).mean(1))
    pv=float((1+(null>=abs(delta.mean())-1e-12).sum())/(len(null)+1))
    summary={'BRB_mean_subject_accuracy_pct':float(df.accuracy.mean()*100),
        'LDA_mean_subject_accuracy_pct':float(df.lda_accuracy.mean()*100),
        'BRB_pooled_accuracy_pct':float(np.average(df.accuracy,weights=df.windows)*100),
        'LDA_pooled_accuracy_pct':float(np.average(df.lda_accuracy,weights=df.windows)*100),
        'BRB_mean_macro_f1_pct':float(df.macro_f1.mean()*100),
        'mean_gain_pp':float(delta.mean()),'gain_subject_bootstrap95_pp':np.quantile(boot,[.025,.975]).tolist(),
        'subject_signflip_p':pv,'subjects_improved':int((delta>0).sum()),'subjects_harmed':int((delta<0).sum())}
    write(out/'summary.json',summary)
    completion={'success':True,'experiment_id':'DB7-031','subjects':list(range(1,21)),'seeds':list(SEEDS),
        'final_brb_fits':60,'validation_brb_fits':240,'test_window_seed_evaluations':int(df.windows.sum()),
        'test_used_for_selection':False,'rules_per_channel':81,'channel_modules':12,'features':48,
        'references_per_feature':3,'all_parameter_groups_updated':True,'devices_used':sorted(devices)}
    assert completion['test_window_seed_evaluations']==695163 and devices=={'cuda:0','cuda:1'}
    write(out/'completion.json',completion)
    report=['# DB7-031 hierarchical BRB results','',
        'Direct EMG-only gesture classifier: 48 features, three references each,12 channel BRBs of81 rules, learned ER fusion weights.',
        'Consequent beliefs, rule weights, attribute weights and fusion weights were jointly trained. Reference values were fitted on training data, then fixed.',
        'Train1/3/4/6, test2/5;200ms;50ms training/10ms test;20 subjects;three seeds;fourfold training repetition validation.',
        'Outer-validation preprocessing uses nested training-only threshold screening; test was not used for selection.',
        'Previously inspected DB7 recordings: exploratory, not independent confirmation. Compare primarily with matched EMG-only LDA; SI also uses inertial inputs.','',
        '```json',json.dumps(summary,indent=2),'```','',
        'Initial/trained rules and weights, references, validation histories, predictions,confusions and per-gesture metrics are stored per subject/seed.']
    (out/'REPORT.md').write_text('\n'.join(report),encoding='utf-8')

def main():
    parser=argparse.ArgumentParser()
    parser.add_argument('--self-test',action='store_true');parser.add_argument('--worker',action='store_true')
    parser.add_argument('--input',type=Path);parser.add_argument('--out',type=Path,default=Path('/kaggle/working/db7_031_results'))
    parser.add_argument('--parent',type=Path,default=Path('/kaggle/working/db7_031_parent.json'))
    parser.add_argument('--device',default='cuda:0');parser.add_argument('--subjects',type=int,nargs='+')
    args=parser.parse_args()
    import torch
    torch.set_num_threads(1)
    if args.self_test:self_test();return
    args.out.mkdir(parents=True,exist_ok=True)
    parent=json.loads(args.parent.read_text());raw=args.input or features.find_root()
    if args.worker:
        torch.cuda.set_device(args.device)
        write(args.out/f'worker_{args.device[-1]}_runtime.json',{'device':args.device,'gpu_name':torch.cuda.get_device_name(args.device),'pid':os.getpid(),'subjects':args.subjects,'torch_version':torch.__version__})
        for subject in args.subjects:run_subject(raw,subject,args.out,parent,args.device)
        return
    assert torch.cuda.device_count()>=2,'Two GPUs required; verify T4 x2 allocation before fitting'
    self_test()
    write(args.out/'PROTOCOL.json',{'experiment_id':'DB7-031','max_epochs':MAX_EPOCHS,'batch_size':BATCH,'optimizer':'Adam','learning_rate':LR,'regularization':PENALTY,
        'train_repetitions':features.TRAIN_REPS,'test_repetitions':features.TEST_REPS,'window_ms':200,'training_stride_ms':50,'test_stride_ms':10,
        'reference_values':'Training feature 5th/50th/95th percentiles, strictly ordered with tiny expansion for ties; fixed after fitting.',
        'architecture':'12 channel BRBs x81 rules;4 antecedents per module;17 consequent classes;trainable-weight ER fusion.',
        'learned':['consequent beliefs','rule weights','attribute weights','fusion weights'],
        'attribute_weight_constraint':'exp(logweight), normalized by channel maximum; raw logweights projected to[-2,2].',
        'rule_weight_constraint':'exp(logweight),logweight in[-2,2]; activations normalized across matching rules.',
        'epoch_selection':'Earliest epoch maximizing mean fourfold training-repetition validation accuracy, up to60; final refit on all4 training reps.',
        'fold_threshold_selection':'Nested LDA ratio screening on the three outer-fitting repetitions only; final threshold is the verified DB7-030 train-only choice.',
        'test_used_for_selection':False,'seeds':SEEDS,'subjects':list(range(1,21)),'neural_fits':0})
    workers=[]
    for device in range(2):
        subjects=list(range(device+1,21,2))
        cmd=[sys.executable,str(Path(__file__).resolve()),'--worker','--device',f'cuda:{device}',
             '--input',str(raw),'--out',str(args.out),'--parent',str(args.parent),'--subjects',*map(str,subjects)]
        workers.append(subprocess.Popen(cmd))
    codes=[p.wait() for p in workers]
    assert codes==[0,0],f'Worker failures {codes}'
    summarize(args.out)
    archive=args.out.parent/'db7_031_results.zip'
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for file in sorted(args.out.rglob('*')):
            if file.is_file():z.write(file,file.relative_to(args.out))
    assert zipfile.ZipFile(archive).testzip() is None
    print('DB7-031 COMPLETE',archive,flush=True)

if __name__=='__main__':main()


## 3. Record verified parent thresholds and raw-data hashes
These choices came only from DB7-030 training repetitions. Parent test accuracy is used as a reproduction check for the LDA comparator, never to tune the BRB.

In [ ]:
%%writefile /kaggle/working/db7_031_parent.json
{
  "1": {
    "raw_emg_sha256": "1c56de390b3f405c9e73f55560aa5ee03c2393769ae76663d9e1cedbc3353240",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      0.00013266204041428864,
      0.0001374756684526801,
      2.5743274818523787e-05,
      1.784747109923046e-05,
      4.47269449068699e-05,
      4.1559411329217255e-05,
      6.860536814201623e-05,
      7.202202687039971e-05,
      4.7443038056371734e-05,
      0.00014688255032524467,
      1.9498520487104543e-05,
      2.1888230548938736e-05
    ],
    "test_windows_10ms": 11553,
    "lda_test_accuracy": 0.7694105427161776
  },
  "2": {
    "raw_emg_sha256": "cd05b892b99337e5c2f12969b7394cdfa09ac1f97108a451db05069089ebd7dc",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      0.0001072564409696497,
      0.00015579882892780006,
      2.5860686946543865e-05,
      1.408601110597374e-05,
      6.473016401287168e-05,
      8.070121111813933e-05,
      7.445613300660625e-05,
      8.223701297538355e-05,
      6.904384645167738e-05,
      0.0001909212296595797,
      1.370580866932869e-05,
      1.9386217900319025e-05
    ],
    "test_windows_10ms": 12695,
    "lda_test_accuracy": 0.6994092162268609
  },
  "3": {
    "raw_emg_sha256": "ed4b01bf9ce41d57fa9a8012d49720085b1c02baf6338bf35e3074c78aa5a9af",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      4.7988301957957447e-05,
      7.463988004019484e-05,
      1.5963290934450924e-05,
      1.139963114837883e-05,
      3.296114664408378e-05,
      2.861838038370479e-05,
      8.237643487518653e-05,
      4.448648905963637e-05,
      3.60179110430181e-05,
      5.011692701373249e-05,
      9.660177056503017e-06,
      7.65162258176133e-06
    ],
    "test_windows_10ms": 10087,
    "lda_test_accuracy": 0.7338158025180926
  },
  "4": {
    "raw_emg_sha256": "8a5a3fa94942dcb3cb0c1ee1b552765f8b8792bf296f6b45ad7217114ebcac0e",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      5.8390131016494706e-05,
      6.557466986123472e-05,
      3.7614106986438856e-05,
      2.1634888980770484e-05,
      2.2602995159104466e-05,
      4.0170711145037785e-05,
      4.2657338781282306e-05,
      3.892793756676838e-05,
      5.607752245850861e-05,
      7.70775877754204e-05,
      2.044211760221515e-05,
      3.140805347356945e-05
    ],
    "test_windows_10ms": 13860,
    "lda_test_accuracy": 0.7521645021645021
  },
  "5": {
    "raw_emg_sha256": "3b0e01332b51804a4da8770d6177e98c02d83fed03954b2a10df827663ad2b84",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      7.318822463275865e-05,
      7.545558037236333e-05,
      1.76858484337572e-05,
      1.4217462194210384e-05,
      6.991653935983777e-05,
      5.3042720537632704e-05,
      8.809279825072736e-05,
      7.663028372917324e-05,
      6.630470306845382e-05,
      7.763763278489932e-05,
      1.476779107179027e-05,
      9.451666301174555e-06
    ],
    "test_windows_10ms": 9278,
    "lda_test_accuracy": 0.7661133865057125
  },
  "6": {
    "raw_emg_sha256": "4df95dd071a2c7630bf1cde146c24009b93a576d8863fe9dccc16ac25dfe8bc8",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      0.00010012800339609385,
      0.00011872818868141621,
      5.414242696133442e-05,
      3.222898885724135e-05,
      0.00011934531357837841,
      0.00017102088895626366,
      8.643615728942677e-05,
      8.578220149502158e-05,
      0.0001707668707240373,
      0.000146279446198605,
      2.7130547096021473e-05,
      1.381685069645755e-05
    ],
    "test_windows_10ms": 11550,
    "lda_test_accuracy": 0.7421645021645021
  },
  "7": {
    "raw_emg_sha256": "acc1ffa3d15e94345e62c23776afb21fc08a7f8888c56fde90e1a630cc3f67d7",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      0.0001515137410024181,
      0.000333660893375054,
      5.334172965376638e-05,
      3.843389276880771e-05,
      0.0002180442970711738,
      0.0001883734221337363,
      0.00015650829300284386,
      0.00013093843881506473,
      0.0001933201856445521,
      0.00031991733703762293,
      1.6768732166383415e-05,
      8.15492239780724e-05
    ],
    "test_windows_10ms": 9383,
    "lda_test_accuracy": 0.7490141745710327
  },
  "8": {
    "raw_emg_sha256": "ce9071373566c7db8bdc0be56e8c645930ac9b47be1b28e80d88e9d55d4028aa",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      0.0001147155198850669,
      0.00027329017757438123,
      3.581086275517009e-05,
      1.6894049622351304e-05,
      6.151852721814066e-05,
      9.829152259044349e-05,
      9.444140596315265e-05,
      6.824731826782227e-05,
      5.348913691705093e-05,
      0.00013976218178868294,
      9.565607797412667e-06,
      1.2080162377969828e-05
    ],
    "test_windows_10ms": 11886,
    "lda_test_accuracy": 0.767794043412418
  },
  "9": {
    "raw_emg_sha256": "bee0940a4ffe3f252fbc07ec3e0e0afd3b67d796e5ae333bc718e2b9f3de07ce",
    "threshold_ratio": 0.005,
    "threshold_scale_final": [
      0.00010828717495314777,
      0.0001047237019520253,
      2.7731381123885512e-05,
      1.8029701095656492e-05,
      0.00011607875057961792,
      9.820485138334334e-05,
      4.905431705992669e-05,
      5.873566624359228e-05,
      7.305116741918027e-05,
      0.00013037046301178634,
      1.738184437272139e-05,
      3.154447040287778e-05
    ],
    "test_windows_10ms": 10305,
    "lda_test_accuracy": 0.7412906356137797
  },
  "10": {
    "raw_emg_sha256": "1e46abf381b0056ac17923dd7612be87c4681eb9c32fc4d7c2f1459fcc7d39c9",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      0.0001176421792479232,
      0.00012255931505933404,
      0.00010596213542157784,
      3.420419670874253e-05,
      9.902081364998594e-05,
      9.533303091302514e-05,
      7.600613025715575e-05,
      7.306517363758758e-05,
      0.00012148161476943642,
      0.00010842792835319415,
      1.7887134163174778e-05,
      2.2611628082813695e-05
    ],
    "test_windows_10ms": 11750,
    "lda_test_accuracy": 0.6695319148936171
  },
  "11": {
    "raw_emg_sha256": "f11d5dfdf6092433e71ab7e88b93ea0810a9e1effd5a6e8ff51b1b2c50d33722",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      5.514318399946205e-05,
      8.086879824986681e-05,
      3.407187614357099e-05,
      1.5211981008178554e-05,
      3.629827551776543e-05,
      2.9886939955758862e-05,
      2.5073355573113076e-05,
      4.232222636346705e-05,
      5.541784412343986e-05,
      5.866068386239931e-05,
      1.6799922377686016e-05,
      1.605762190592941e-05
    ],
    "test_windows_10ms": 14244,
    "lda_test_accuracy": 0.689623701207526
  },
  "12": {
    "raw_emg_sha256": "afa6eb0517c388c28f458886332cb96be4469e3376731aaced584f1b9594b56f",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      0.00010118428326677531,
      0.00021223636576905847,
      3.55233351001516e-05,
      1.3748856872553006e-05,
      8.230475941672921e-05,
      0.0001146679132943973,
      0.00011095914669567719,
      6.208808190422133e-05,
      9.482594032306224e-05,
      9.964436321752146e-05,
      7.21825344953686e-05,
      9.099677299673203e-06
    ],
    "test_windows_10ms": 12272,
    "lda_test_accuracy": 0.6676988265971316
  },
  "13": {
    "raw_emg_sha256": "0e55d064adfe43be83738eb22ab99bcfa19fda45bbd0048c1bac382575e89282",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      5.280390178086236e-05,
      9.842139843385667e-05,
      6.807075988035649e-05,
      1.2225775208207779e-05,
      2.12231716432143e-05,
      9.242261876352131e-05,
      5.533037619898096e-05,
      5.869295273441821e-05,
      9.206052345689386e-05,
      0.00011729584366548806,
      3.182988439220935e-05,
      8.137374607031234e-06
    ],
    "test_windows_10ms": 12707,
    "lda_test_accuracy": 0.6941056110805068
  },
  "14": {
    "raw_emg_sha256": "602afca1f5de11d0ad728f4414394f38d0c1d90659d4507b692c97a9ccc54311",
    "threshold_ratio": 0.01,
    "threshold_scale_final": [
      0.00013264398148749024,
      0.00021044100867584348,
      4.297760460758582e-05,
      1.7475626009400003e-05,
      0.00011667045328067616,
      7.852225826354697e-05,
      0.00011536300007719547,
      0.00016484233492519706,
      0.00011176728730788454,
      8.786814578343183e-05,
      2.2566648112842813e-05,
      1.3923663573223166e-05
    ],
    "test_windows_10ms": 11584,
    "lda_test_accuracy": 0.7045925414364641
  },
  "15": {
    "raw_emg_sha256": "bf717ea71162a5cf01445393c3a20d66bf3af2b21cf8d926161d9a34a8ee3c28",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      6.782060518162325e-05,
      0.00016464098007418215,
      2.5319724954897538e-05,
      1.2249682185938582e-05,
      5.61313281650655e-05,
      3.773471325985156e-05,
      3.8305443013086915e-05,
      3.9486305468017235e-05,
      6.851983198430389e-05,
      5.4393927712226287e-05,
      1.3233364370535128e-05,
      1.5889450878603384e-05
    ],
    "test_windows_10ms": 11455,
    "lda_test_accuracy": 0.7030117852466172
  },
  "16": {
    "raw_emg_sha256": "c02de75b4bc2ede5715b71d704f4969ddfef14de442e407b6559991c898a0868",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      0.00019861815962940454,
      0.00016362268070224673,
      7.93134022387676e-05,
      2.511195998522453e-05,
      3.445372567512095e-05,
      6.207184924278408e-05,
      7.871009438531473e-05,
      0.00015840321430005133,
      8.483541023451835e-05,
      0.00017966260202229023,
      1.3469390978571028e-05,
      1.391061505273683e-05
    ],
    "test_windows_10ms": 11385,
    "lda_test_accuracy": 0.7006587615283267
  },
  "17": {
    "raw_emg_sha256": "3c819fbca8446569bd2c31fb49959d7796a49d0762304f24f8134af73eb73317",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      5.337742550182156e-05,
      0.0001356772263534367,
      5.898009112570435e-05,
      1.3639853023050819e-05,
      3.406200994504616e-05,
      5.478320235852152e-05,
      5.0377362640574574e-05,
      2.9153767172829248e-05,
      5.622144090011716e-05,
      0.00012032186350552365,
      2.2048187020118348e-05,
      8.751561836106703e-06
    ],
    "test_windows_10ms": 11039,
    "lda_test_accuracy": 0.7352115227828607
  },
  "18": {
    "raw_emg_sha256": "f6a6081a3fc28fc7ac0b5e40b82e1206cf881d3c1ee4fb8192772cce773bdf93",
    "threshold_ratio": 0.02,
    "threshold_scale_final": [
      6.631188443861902e-05,
      8.188419451471418e-05,
      2.2324860765365884e-05,
      9.289688023272902e-06,
      1.9854758647852577e-05,
      4.6616514737252146e-05,
      2.153000059479382e-05,
      5.2621340728364885e-05,
      4.335492121754214e-05,
      4.028893090435304e-05,
      9.028473868966103e-06,
      1.9218265151721425e-05
    ],
    "test_windows_10ms": 13052,
    "lda_test_accuracy": 0.7860098069261416
  },
  "19": {
    "raw_emg_sha256": "3505bf92ff6501279405b4b6988a972f54bfe7493e3f3f88c19eef571ab7cd8a",
    "threshold_ratio": 0.005,
    "threshold_scale_final": [
      9.851076902123168e-05,
      0.0001474832242820412,
      3.4470394894015044e-05,
      1.9086524844169617e-05,
      5.6170196330640465e-05,
      8.336344035342336e-05,
      9.021452569868416e-05,
      5.269997564028017e-05,
      4.7443387302337214e-05,
      8.203284232877195e-05,
      2.193837644881569e-05,
      1.8382097550784238e-05
    ],
    "test_windows_10ms": 10975,
    "lda_test_accuracy": 0.6433712984054669
  },
  "20": {
    "raw_emg_sha256": "58bb627d65f3cd87a3c434f781126589eea47d19727cc20e54c5b701ad6ddf72",
    "threshold_ratio": 0.01,
    "threshold_scale_final": [
      0.00014526289305649698,
      0.00019448946113698184,
      0.0004191744374111295,
      4.356720091891475e-05,
      2.549423516029492e-05,
      0.00016596984642092139,
      0.00010782211029436439,
      6.814220250817016e-05,
      0.0001349761732853949,
      0.0001800787722459063,
      3.6949502828065306e-05,
      2.0891255189781077e-05
    ],
    "test_windows_10ms": 10661,
    "lda_test_accuracy": 0.6287402682675172
  }
}

## 4. Train on both GPUs and verify completion
Initial/trained rule beliefs, rule weights, attribute weights, fusion weights and raw reference values are exported per subject/seed. Test predictions are saved only after epoch selection and final fitting.

In [ ]:
import os,sys,subprocess,json
from pathlib import Path
os.environ['OPENBLAS_NUM_THREADS']='1'
os.environ['OMP_NUM_THREADS']='1'
subprocess.run([sys.executable,'/kaggle/working/db7_031_brb.py'],check=True)
out=Path('/kaggle/working/db7_031_results')
done=json.loads((out/'completion.json').read_text())
assert done['success'] and done['final_brb_fits']==60 and done['validation_brb_fits']==240
assert done['devices_used']==['cuda:0','cuda:1'] and not done['test_used_for_selection']
assert done['test_window_seed_evaluations']==695163 and done['all_parameter_groups_updated']
print(json.dumps(done,indent=2))


## 5. Present measured accuracy and subject/gesture diagnostics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,Markdown
display(Markdown((out/'REPORT.md').read_text()))
metrics=pd.read_csv(out/'subject_seed_metrics.csv')
display(metrics)
by_subject=metrics.groupby('subject')[['accuracy','lda_accuracy']].mean()*100
ax=by_subject.plot.bar(figsize=(14,5),ylim=(0,100),ylabel='Test accuracy (%)',title='Hierarchical BRB versus matched Hudgins LDA')
plt.tight_layout();plt.savefig(out/'subject_accuracy.png',dpi=160);plt.show()
g=pd.read_csv(out/'subject_gesture_metrics.csv')
grid=g.groupby(['subject','gesture']).recall.mean().unstack()*100
fig,ax=plt.subplots(figsize=(12,7));im=ax.imshow(grid,vmin=0,vmax=100,aspect='auto',cmap='viridis')
ax.set(xticks=range(17),xticklabels=range(1,18),yticks=range(20),yticklabels=range(1,21),xlabel='Gesture',ylabel='Subject',title='BRB mean recall by subject and gesture')
fig.colorbar(im,ax=ax,label='Recall (%)');fig.tight_layout();fig.savefig(out/'gesture_recall.png',dpi=160);plt.show()
# Refresh the archive to include these plots.
import zipfile
with zipfile.ZipFile('/kaggle/working/db7_031_results.zip','w',zipfile.ZIP_DEFLATED,compresslevel=6) as archive:
    for file in sorted(out.rglob('*')):
        if file.is_file():archive.write(file,file.relative_to(out))
print('Download db7_031_results.zip for all models, rules, weights and predictions.')
